In [129]:

from DATA.stock_invest_function import *

import pymysql
import pandas as pd
from typing import Dict
from typing import Dict, Optional
from typing import Tuple


def fetch_roe_roa_from_db(db_info: Dict,
                          ticker: str,
                          indicator: str,
                          table_name: str = "korea_fs_data_roe_roa") -> pd.DataFrame:
    """
    investar.korea_fs_data_roe_roa 에서
    특정 ticker + indicator 의 시계열 데이터를 가져오는 함수.
    """
    # db / database 둘 다 허용
    db_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_name,
        charset="utf8mb4"
    )

    try:
        sql = f"""
            SELECT date, ticker, indicator, value
            FROM {table_name}
            WHERE ticker = %s
              AND indicator = %s
            ORDER BY date
        """
        df = pd.read_sql(sql, conn, params=[ticker, indicator])
    finally:
        conn.close()

    if not df.empty:
        df["date"] = pd.to_datetime(df["date"])
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

    return df


def fetch_required_return_quarterly_avg(db_info: Dict,
                                        ticker: str,
                                        table_name: str = "korea_required_return_result") -> pd.DataFrame:
    """
    특정 ticker 의 required_return 을 불러와
    분기별 평균 required_return 을 계산.
    """
    db_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_name,
        charset="utf8mb4"
    )

    try:
        sql = f"""
            SELECT date, ticker, indicator, value
            FROM {table_name}
            WHERE ticker = %s
              AND indicator = 'required_return'
            ORDER BY date
        """
        df = pd.read_sql(sql, conn, params=[ticker])
    finally:
        conn.close()

    if df.empty:
        return df

    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["value"])

    quarterly = (
        df
        .set_index("date")
        .groupby([pd.Grouper(freq="Q"), "ticker"])["value"]
        .mean()
        .reset_index()
        .rename(columns={"value": "required_return_q_avg"})
    )

    return quarterly

from typing import Dict, Optional


def fetch_latest_equity(db_info: Dict,
                        ticker: str,
                        table_name: str = "korea_fs_data_roe_roa") -> Optional[float]:
    """
    특정 ticker의 '자본총계' indicator의 가장 최근 value 값을 반환하는 함수.
    값이 없거나 존재하지 않으면 None 반환.
    """

    # db 또는 database 키 모두 지원
    db_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_name,
        charset="utf8mb4"
    )

    try:
        sql = f"""
            SELECT date, value
            FROM {table_name}
            WHERE ticker = %s
              AND indicator = '자본총계'
            ORDER BY date DESC
            LIMIT 1
        """
        df = pd.read_sql(sql, conn, params=[ticker])
    finally:
        conn.close()

    if df.empty:
        return None

    # value 숫자 변환
    value = pd.to_numeric(df.loc[0, "value"], errors="coerce")

    return value

import pymysql
import pandas as pd
from typing import Dict


def fetch_yearly_avg_payout_ratio(db_info: Dict,
                                  ticker: str,
                                  table_name: str = "korea_fs_data_roe_roa") -> pd.DataFrame:
    """
    특정 ticker의 payout_ratio 값을 DB에서 불러와
    절대값으로 변환한 뒤,
    연간 평균 payout_ratio를 계산하여 반환.

    반환 DataFrame:
        year | ticker | payout_ratio_yearly_avg
    """
    # db 또는 database 키 모두 지원
    db_name = db_info.get("db") or db_info.get("database")

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_name,
        charset="utf8mb4"
    )

    try:
        sql = f"""
            SELECT date, value
            FROM {table_name}
            WHERE ticker = %s
              AND indicator = 'payout_ratio'
            ORDER BY date
        """
        df = pd.read_sql(sql, conn, params=[ticker])
    finally:
        conn.close()

    if df.empty:
        return df

    # 날짜 변환
    df["date"] = pd.to_datetime(df["date"])

    # 숫자 변환
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    # 절대값 처리
    df["value"] = df["value"].abs()

    # 연도 정보 생성
    df["year"] = df["date"].dt.year

    # 연간 평균 계산
    yearly_avg = (
        df.groupby("year")["value"]
        .mean()
        .reset_index()
        .rename(columns={"value": "payout_ratio_yearly_avg"})
    )

    # ticker 추가
    yearly_avg["ticker"] = ticker

    # 순서 정리
    yearly_avg = yearly_avg[["year", "ticker", "payout_ratio_yearly_avg"]]

    return yearly_avg


In [130]:
# 예시 db_info
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

# 1) ROE_ttm 시계열
roe_df = fetch_roe_roa_from_db(db_info, ticker="000660", indicator="ROE")

# 2) 000660 분기별 required_return 평균
req_q_df = fetch_required_return_quarterly_avg(db_info, ticker="000660")

roe_pivot = (
    roe_df.pivot_table(
        index="date",
        columns="indicator",
        values="value",
        aggfunc="first"     # indicator는 중복 없으므로 첫 값 사용
    )
    .sort_index()
)

req_q_df = req_q_df.copy()

# 1) date를 datetime + index 설정
req_q_df["date"] = pd.to_datetime(req_q_df["date"])
# req_q_df = req_q_df.set_index("date").sort_index()

# 2) required_return_q_avg를 100으로 나누기
req_q_df["required_return_q_avg"] = req_q_df["required_return_q_avg"] / 100

ri_raw = pd.merge(roe_pivot, req_q_df, on="date", how="inner")
ri_raw['residual_rate'] = ri_raw['ROE'] - ri_raw['required_return_q_avg']

latest_equity = fetch_latest_equity(db_info, ticker="000660")
# e단위는 억이다
book_value = latest_equity/100000

re_mean = ri_raw['required_return_q_avg'].mean()
roe_mean = ri_raw['ROE'].mean()

yearly_payout = fetch_yearly_avg_payout_ratio(db_info, ticker="000660")

# div_mean

In [140]:
def residual_income_valuation(
    initial_book_value: float,
    current_roe: float,
    required_return: float,
    payout_ratio: float = 0.05,
    n_high_years: int = 3,
    n_fade_years: int = 7,
) -> Tuple[pd.DataFrame, float]:
    """
    Residual Income Model로 기업 가치를 계산.

    Parameters
    ----------
    initial_book_value : float
        t=0 시점의 기초 자기자본 (B0)
    current_roe : float
        현재 ROE (예: 0.30 = 30%)
    required_return : float
        요구수익률 re (예: 0.10727 = 10.727%), 할인율도 이 값 사용
    payout_ratio : float, default 0.05
        배당성향 (NI 중 배당 비율)
    n_high_years : int, default 3
        현재 ROE가 유지되는 기간 (년 수)
    n_fade_years : int, default 7
        ROE가 점진적으로 required_return 수준까지 떨어지는 기간 (년 수)

    Returns
    -------
    ri_df : pd.DataFrame
        연도별 Book Value, ROE, re, (ROE - re), NI, Residual Income,
        Dividend, Ending Book, Present Value of RI 등을 담은 표
    intrinsic_value : float
        Residual Income Model로 계산한 적정 가치 (B0 + PV(RI))
    """

    total_years = n_high_years + n_fade_years

    # 1) ROE 경로 생성
    # 1~n_high_years: current_roe 유지
    high_roe = np.full(n_high_years, current_roe)

    # 이후 n_fade_years 동안 current_roe -> required_return 선형하락
    # 예: [0.30, ..., 0.10727] 중 첫 번째를 버리고 7개만 사용
    fade_roe = np.linspace(current_roe, required_return, n_fade_years + 1)[1:]

    roe_path = np.concatenate([high_roe, fade_roe])

    # 2) 연도별 계산
    records = []
    B_t = float(initial_book_value)

    for year in range(1, total_years + 1):
        roe_t = float(roe_path[year - 1])
        re_t = float(required_return)

        # 당기순이익
        ni_t = B_t * roe_t

        # 배당
        div_t = ni_t * payout_ratio

        # Residual Income
        spread_t = roe_t - re_t      # (ROE - re)
        ri_t = B_t * spread_t        # RI_t = B_{t-1} * (ROE_t - re)

        # 기말 자기자본
        end_B = B_t + ni_t - div_t   # B_t + (1 - payout)*NI_t

        # 현재가치
        discount_factor = 1 / ((1 + re_t) ** year)
        pv_ri_t = ri_t * discount_factor

        records.append({
            "year": year,
            "begin_book": B_t,
            "roe": roe_t,
            "required_return": re_t,
            "spread_roe_minus_re": spread_t,
            "net_income": ni_t,
            "dividend": div_t,
            "residual_income": ri_t,
            "end_book": end_B,
            "discount_factor": discount_factor,
            "pv_residual_income": pv_ri_t,
        })

        # 다음 해를 위해 갱신
        B_t = end_B

    ri_df = pd.DataFrame(records)

    # 3) 적정 가치 계산: V0 = B0 + Σ PV(RI_t)
    intrinsic_value = initial_book_value + ri_df["pv_residual_income"].sum()

    return ri_df, intrinsic_value




def rim_constant_spread_with_payout(
    initial_book_value: float,
    required_return: float,
    spread: float = 0.06,
    payout_ratio: float = 0.05,
    horizon_years: int = 30,
) -> Tuple[pd.DataFrame, float]:
    """
    Residual Income Model with:
    - constant spread (ROE = re + spread)
    - payout ratio (<100%)
    - growing book value

    Parameters
    ----------
    initial_book_value : float
    required_return : float (e.g., 0.10)
    spread : float (e.g., 0.06)
    payout_ratio : float (e.g., 0.05 → 5%)
    horizon_years : int

    Returns
    -------
    ri_df : pd.DataFrame (detailed calculation sheet)
    intrinsic_value : float (V0 = B0 + Σ PV(RI_t))
    """

    B_t = float(initial_book_value)
    re = float(required_return)
    s = float(spread)
    payout = float(payout_ratio)

    roe = re + s

    records = []

    for t in range(1, horizon_years + 1):
        # 순이익
        ni = B_t * roe

        # 배당
        dividend = ni * payout

        # Residual Income
        ri = B_t * s

        # 할인
        discount_factor = 1 / ((1 + re) ** t)
        pv_ri = ri * discount_factor

        # 기말 자본
        end_book = B_t + ni - dividend

        records.append({
            "year": t,
            "begin_book": B_t,
            "roe": roe,
            "required_return": re,
            "spread_roe_minus_re": s,
            "net_income": ni,
            "dividend": dividend,
            "residual_income": ri,
            "end_book": end_book,
            "discount_factor": discount_factor,
            "pv_residual_income": pv_ri,
        })

        B_t = end_book  # 다음 해로 업데이트

    ri_df = pd.DataFrame(records)

    intrinsic_value = initial_book_value + ri_df["pv_residual_income"].sum()

    return ri_df, intrinsic_value


In [141]:
# 예: 기초 자기자본 (원 단위, 예시 숫자)
B0 = book_value  # 5조원 같은 식으로 입력

# ri_raw에서 마지막 요구수익률 (예: 0.107270)
re_last = float(ri_raw["required_return_q_avg"].iloc[-1])

# 현재 ROE 30% 가정
current_roe = 0.30

ri_table, fair_value = residual_income_valuation(
    initial_book_value=B0,
    current_roe=current_roe,
    required_return=re_last,
    payout_ratio=0.05,
    n_high_years=5,
    n_fade_years=5,
)

print(ri_table)       # 연간 residual income, book value, (roe - re) 확인
print("적정 가치 V0:", fair_value)

   year    begin_book       roe  required_return  spread_roe_minus_re  \
0     1  1.000063e+06  0.300000          0.10727             0.192730   
1     2  1.285081e+06  0.300000          0.10727             0.192730   
2     3  1.651329e+06  0.300000          0.10727             0.192730   
3     4  2.121957e+06  0.300000          0.10727             0.192730   
4     5  2.726715e+06  0.300000          0.10727             0.192730   
5     6  3.503829e+06  0.261454          0.10727             0.154184   
6     7  4.374114e+06  0.222908          0.10727             0.115638   
7     8  5.300388e+06  0.184362          0.10727             0.077092   
8     9  6.228717e+06  0.145816          0.10727             0.038546   
9    10  7.091550e+06  0.107270          0.10727             0.000000   

      net_income      dividend  residual_income      end_book  \
0  300018.837000  15000.941850    192742.400186  1.285081e+06   
1  385524.205545  19276.210277    247673.984239  1.651329e+06   
2

In [142]:
longterm_g = roe_mean - re_mean

# 예시 숫자 — 실제 값은 교수님이 입력
B0 = book_value  # SK하이닉스 자기자본 (원) 예시
rr = 0.10                # 요구수익률 10% 가정
spread = longterm_g           # (ROE - re) 장기 평균 6%

ri_df, fair_value = rim_constant_spread_with_payout(
    initial_book_value= B0,
    required_return=rr,
    spread=spread,
    payout_ratio=0.05,   # ← 여기 숫자만 바꾸면 됩니다 (5% 배당)
    horizon_years=30
)

In [143]:
print(ri_table)


   year    begin_book       roe  required_return  spread_roe_minus_re  \
0     1  1.000063e+06  0.300000          0.10727             0.192730   
1     2  1.285081e+06  0.300000          0.10727             0.192730   
2     3  1.651329e+06  0.300000          0.10727             0.192730   
3     4  2.121957e+06  0.300000          0.10727             0.192730   
4     5  2.726715e+06  0.300000          0.10727             0.192730   
5     6  3.503829e+06  0.261454          0.10727             0.154184   
6     7  4.374114e+06  0.222908          0.10727             0.115638   
7     8  5.300388e+06  0.184362          0.10727             0.077092   
8     9  6.228717e+06  0.145816          0.10727             0.038546   
9    10  7.091550e+06  0.107270          0.10727             0.000000   

      net_income      dividend  residual_income      end_book  \
0  300018.837000  15000.941850    192742.400186  1.285081e+06   
1  385524.205545  19276.210277    247673.984239  1.651329e+06   
2

In [144]:
fair_value

4891755.566110822

In [252]:
cont_ops_df

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker


In [195]:
df.to_excel(r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Results\Korea\fs_sample.xlsx')